# SUPP: Summary of participant-reported sample ratios.

```
SUPP: Summary of participant-reported sample ratios.

Box-and-whisker of the A/B, B/C, A/C ratios each participant REPORTED in their
*-sampleRatios file, by species, with dashed design-expected lines. (The companion
recalculated-from-peptide-quant version is FIG03B.)

CURATION (examined every file in each subdirectory):
  * Exclude 05, 42 (only '*this is fake data' placeholder sampleRatios).
  * Site 02: .tsv is the fake template -> use 02-sampleRatios.xlsx.
  * Site 03: plain .tsv is fake -> use 03-sampleRatios_completed.tsv.
  * Site 14: Templates/ copy is fake -> use the two real workflow variants (14A/14B).
  * Multi-workflow labs kept as A/B: 07A/07B, 10A/10B, 14A/14B.
```

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

DATA_DIR = r"D:/2022 Multi-Species Standard Study"
OUTPUT = "output"
DATA = "data"
os.makedirs(OUTPUT, exist_ok=True)
os.makedirs(DATA, exist_ok=True)

CURATED = {   # label -> reported sampleRatios file
    "01": "01/01-sampleRatios.tsv",
    "02": "02/02-sampleRatios.xlsx",                 # .tsv is fake
    "03": "03/03-sampleRatios_completed.tsv",        # plain .tsv is fake
    "04": "04/04-sampleRatios.tsv",
    "06": "06/06-sampleRatios.tsv",
    "07A": "07/07-sampleRatios.tsv",
    "07B": "07/07-sampleRatios_ShortGrad.tsv",
    "09": "09/09-sampleRatios.tsv",
    "10A": "10/10-QEx_sampleRatios.tsv",
    "10B": "10/10-Fusion_sampleRatios.tsv",
    "11": "11/11-sampleRatios.tsv",
    "12": "12/12-sampleRatios.tsv",
    "14A": "14/14-sampleRatios_sPRG_lumos_DIA_1x8mzStag_3x4mzGPFlibrary.txt",
    "14B": "14/14-sampleRatios_sPRG_lumos_DIA_2x4mzStag_Prosit_library.txt",
    "41": "41/41-sampleRatios.tsv",
}
SPECIES_MAP = {"cow": "Bovine", "bovin": "Bovine", "bovine": "Bovine", "bos taurus": "Bovine",
               "human": "Human", "homo sapiens": "Human",
               "trout": "Trout", "salvelinus namaycush": "Trout", "salnm": "Trout"}
SPECIES_ORDER = ["Bovine", "Human", "Trout"]
COLORS = {"Human": "#E69F00", "Bovine": "#009E73", "Trout": "#56B4E9"}  # human orange, cow green, trout blue
COMPARISONS = ["A/B", "B/C", "A/C"]
MIX = {"A": {"Trout": 50, "Human": 45, "Bovine": 5},
       "B": {"Trout": 50, "Human": 20, "Bovine": 30},
       "C": {"Trout": 50, "Human": 3,  "Bovine": 47}}
EXPECTED = {c: {sp: MIX[c[0]][sp] / MIX[c[-1]][sp] for sp in SPECIES_ORDER} for c in COMPARISONS}

In [2]:
def _read_ratios(path):
    """Keep only the first 4 columns (Sample, A/B, B/C, A/C); tolerate ragged rows."""
    if path.lower().endswith((".xlsx", ".xls")):
        df = pd.read_excel(path).iloc[:, :4]
        df.columns = ["Sample", "A/B", "B/C", "A/C"]
        return df
    with open(path, encoding="utf-8", errors="ignore") as fh:
        lines = [ln.rstrip("\n") for ln in fh if ln.strip()]
    recs = [(ln.split("\t") + ["", "", "", ""])[:4] for ln in lines[1:]]
    return pd.DataFrame(recs, columns=["Sample", "A/B", "B/C", "A/C"])

In [3]:
def load_reported():
    rows = []
    for label, sr in CURATED.items():
        df = _read_ratios(os.path.join(DATA_DIR, sr))
        df["species"] = df["Sample"].map(lambda x: SPECIES_MAP.get(str(x).strip().lower()))
        df = df.dropna(subset=["species"])
        for comp in COMPARISONS:
            for _, r in df.iterrows():
                val = pd.to_numeric(r[comp], errors="coerce")
                if pd.notna(val):
                    rows.append(dict(label=label, species=r["species"], comparison=comp, ratio=float(val)))
    return pd.DataFrame(rows)

In [4]:
def boxwhisker(p, out_png):
    fig, axes = plt.subplots(3, 1, figsize=(5.4, 7.8), sharex=True)
    for ax, comp in zip(axes, COMPARISONS):
        subc = p[p["comparison"] == comp]
        data = [subc.loc[subc["species"] == sp, "ratio"].values for sp in SPECIES_ORDER]
        ax.boxplot(data, orientation="horizontal", tick_labels=SPECIES_ORDER, widths=0.6, showfliers=False)
        ax.set_xscale("log")
        rng = np.random.default_rng(0)
        for i, sp in enumerate(SPECIES_ORDER, start=1):
            ss = subc[subc["species"] == sp]
            if not ss.empty:
                y = np.full(len(ss), i) + rng.uniform(-0.09, 0.09, size=len(ss))
                ax.scatter(ss["ratio"], y, s=18, alpha=0.8, color=COLORS[sp], edgecolor="black", linewidth=0.3, zorder=3)
            ax.axvline(EXPECTED[comp][sp], ls="--", lw=1.1, color=COLORS[sp], zorder=1)
        ax.set_title(comp, fontsize=11); ax.set_xlim(0.02, 30); ax.grid(True, axis="x", linewidth=0.3)
    axes[-1].set_xlabel("Reported ratio (log scale) — dashed = design-expected")
    fig.suptitle("SUPP  Participant-reported ratios (n=%d)" % p["label"].nunique(), fontsize=12)
    fig.tight_layout(rect=(0, 0, 1, 0.97))
    fig.savefig(out_png, dpi=200); fig.savefig(out_png.replace(".png", ".pdf")); plt.close(fig)
    print(f"  wrote {out_png}")

In [5]:
# ---- generate figures ----
rep = load_reported()
rep.to_csv(os.path.join(DATA, "supp_reported_ratios_long.csv"), index=False)
print(f"{len(rep)} points, {rep['label'].nunique()} submissions")
boxwhisker(rep, os.path.join(OUTPUT, "SUPP_reported_ratios.png"))

135 points, 15 submissions


  wrote output\SUPP_reported_ratios.png
